**Loan-Approval-Prediction-Dataset using Decision Tree**

In [ ]:
#Iport Necessary Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
import joblib

**Description of Dataset**

The loan approval dataset is a collection of financial records and associated information used to determine the eligibility of individuals or organizations for obtaining loans from a lending institution. It includes various factors such as cibil score, income, employment status, loan term, loan amount, assets value, and loan status.

In [ ]:
# Load Dataset from my Google Drive
df = pd.read_csv("/content/drive/MyDrive/Machine Learning by Zafar Iqbal/loan_approval_dataset.csv")
df.head(10)

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected
5,6,0,Graduate,Yes,4800000,13500000,10,319,6800000,8300000,13700000,5100000,Rejected
6,7,5,Graduate,No,8700000,33000000,4,678,22500000,14800000,29200000,4300000,Approved
7,8,2,Graduate,Yes,5700000,15000000,20,382,13200000,5700000,11800000,6000000,Rejected
8,9,0,Graduate,Yes,800000,2200000,20,782,1300000,800000,2800000,600000,Approved
9,10,5,Not Graduate,No,1100000,4300000,10,388,3200000,1400000,3300000,1600000,Rejected


In [ ]:
# Checking - How many no. of rows and no. of columns
df.shape

(4269, 13)

In [ ]:
# Checking - Is there exists null values in the dataset or not?
df[df.isnull().any(axis=1)].head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status


**Data Cleaning**

In [ ]:
# Delete the columns : loan_id - no_of_dependents - cibil_score
df = df.drop(df.columns[[0,1,7]],axis=1)
df.head(5)

,education,self_employed,income_annum,loan_amount,loan_term,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,Graduate,No,9600000,29900000,12,2400000,17600000,22700000,8000000,Approved
1,Not Graduate,Yes,4100000,12200000,8,2700000,2200000,8800000,3300000,Rejected
2,Graduate,No,9100000,29700000,20,7100000,4500000,33300000,12800000,Rejected
3,Graduate,No,8200000,30700000,8,18200000,3300000,23300000,7900000,Rejected
4,Not Graduate,Yes,9800000,24200000,20,12400000,8200000,29400000,5000000,Rejected


In [ ]:
# Delete the rows which have any missing value
df.dropna(axis=0,inplace=True)

In [ ]:
# Checking Again  - How many no. of rows and no. of columns
df.shape

(4269, 10)

In [ ]:
# Remove spaces in the column names/headings
df.columns = df.columns.str.strip()

In [ ]:
# Print the Categories of column "self_employed" with numbers
print(df['self_employed'].value_counts())

self_employed
Yes    2150
No     2119
Name: count, dtype: int64


In [ ]:
# Print the Categories of column "education" with numbers
print(df['education'].value_counts())

education
Graduate        2144
Not Graduate    2125
Name: count, dtype: int64


In [ ]:
# Create New Columns
le_education = LabelEncoder()
le_self_employed = LabelEncoder()

# Replace discreate values with 0 and 1
df['education_n'] = le_education.fit_transform(df['education'])
df['self_employed_n'] = le_self_employed.fit_transform(df['self_employed'])
df.head(5)

,education,self_employed,income_annum,loan_amount,loan_term,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,education_n,self_employed_n
0,Graduate,No,9600000,29900000,12,2400000,17600000,22700000,8000000,Approved,0,0
1,Not Graduate,Yes,4100000,12200000,8,2700000,2200000,8800000,3300000,Rejected,1,1
2,Graduate,No,9100000,29700000,20,7100000,4500000,33300000,12800000,Rejected,0,0
3,Graduate,No,8200000,30700000,8,18200000,3300000,23300000,7900000,Rejected,0,0
4,Not Graduate,Yes,9800000,24200000,20,12400000,8200000,29400000,5000000,Rejected,1,1


In [ ]:
# Now Remove the un-necessary columns
df = df.drop(df.columns[[0,1]],axis=1)
df.head(5)

,income_annum,loan_amount,loan_term,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,education_n,self_employed_n
0,9600000,29900000,12,2400000,17600000,22700000,8000000,Approved,0,0
1,4100000,12200000,8,2700000,2200000,8800000,3300000,Rejected,1,1
2,9100000,29700000,20,7100000,4500000,33300000,12800000,Rejected,0,0
3,8200000,30700000,8,18200000,3300000,23300000,7900000,Rejected,0,0
4,9800000,24200000,20,12400000,8200000,29400000,5000000,Rejected,1,1


**Selection of Variables for Model : x = input/features & output/label**

In [ ]:
# Print the Column Names/headings
df.columns

Index(['income_annum', 'loan_amount', 'loan_term', 'residential_assets_value',
       'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value',
       'loan_status', 'education_n', 'self_employed_n'],
      dtype='object')

In [ ]:
x = df[['income_annum', 'loan_amount', 'loan_term', 'residential_assets_value',
       'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value',
        'education_n', 'self_employed_n']]
y = df['loan_status']

In [ ]:
# print x variable
x.head()

,income_annum,loan_amount,loan_term,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,education_n,self_employed_n
0,9600000,29900000,12,2400000,17600000,22700000,8000000,0,0
1,4100000,12200000,8,2700000,2200000,8800000,3300000,1,1
2,9100000,29700000,20,7100000,4500000,33300000,12800000,0,0
3,8200000,30700000,8,18200000,3300000,23300000,7900000,0,0
4,9800000,24200000,20,12400000,8200000,29400000,5000000,1,1


In [ ]:
#print y variable
y.head(5)

,loan_status
0,Approved
1,Rejected
2,Rejected
3,Rejected
4,Rejected


**Split the Dataset into train and test by 80/20 Rule**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=324)

**Create & Fit the Model Using Decision Tree Algorithm**

In [ ]:
model = DecisionTreeClassifier(max_leaf_nodes=10,random_state=0)
model.fit(x_train, y_train)

DecisionTreeClassifier(max_leaf_nodes=10, random_state=0)

**Predicting values on test set of data**

In [ ]:
predicted_values = model.predict(x_test)

**Checking Test Accuracy**

In [ ]:
print("Accuracy Score" , accuracy_score(y_test, predicted_values))
print("Precision Score" , precision_score(y_test, predicted_values, average='micro'))
print("Recall Score" , recall_score(y_test, predicted_values, average='micro'))
print("F1 Score" , f1_score(y_test, predicted_values,average='micro'))

Accuracy Score 0.6428571428571429
Precision Score 0.6428571428571429
Recall Score 0.6428571428571429
F1 Score 0.6428571428571429


**To Save the Model**

In [ ]:
joblib.dump(model, "/content/drive/MyDrive/Machine Learning by Zafar Iqbal/loan_approval_model.joblib ")

['/content/drive/MyDrive/Machine Learning by Zafar Iqbal/loan_approval_model.joblib ']

**To Load the Model**

In [4]:
import joblib
joblib.load("/content/drive/MyDrive/Machine Learning by Zafar Iqbal/loan_approval_model.joblib ")

DecisionTreeClassifier(max_leaf_nodes=10, random_state=0)

**To Make User Interface (UI) Using Gradio**

In [5]:
!pip install gradio

In [6]:
import gradio as gr
def loan_approval(
        income_annum,
        loan_amount,
        loan_term,
        residential_assets_value,
        commercial_assets_value,
        luxury_assets_value,
        bank_asset_value,
        education_n,
        self_employed_n
        ):

    # Convert Catogorial Variables into numbers
    education_n = 1 if education_n == "Graduate" else 0
    self_employed_n =1 if self_employed_n == "Yes" else 0


    data  = pd.DataFrame({
        'income_annum':[income_annum],
        'loan_amount':[loan_amount],
        'loan_term':[loan_term],
        'residential_assets_value':[residential_assets_value],
        'commercial_assets_value':[commercial_assets_value],
        'luxury_assets_value':[luxury_assets_value],
        'bank_asset_value':[bank_asset_value],
        'education_n':[education_n],
        'self_employed_n':[self_employed_n]
    })

    prediction = model.predict(data)

    if prediction[0]==1:
          return "Loan Approved"
    else:
         return "Loan not Approved"

iface = gr.Interface(
    fn = loan_approval,

    inputs=[
        gr.Number(label = 'income per annum'),
        gr.Number(label ='loan amount'),
        gr.Number(label = 'No. of Years to Return Loan'),
        gr.Number(label = 'Value of Residential Assets'),
        gr.Number(label = 'Value ofCcommercial Assets'),
        gr.Number(label = 'Value of Luxury Assets'),
        gr.Number(label = 'Value of Bank Asset'),
        gr.Dropdown(['Graduate' , 'Not Graduate'], label = 'Education'),
        gr.Dropdown(['Yes' , 'No'], label = 'Self Employed')
        ],

    outputs = gr.Textbox(label = 'Loan Status'),
    title = "Loan Approval Prediction App",
    description = "Enter Loan Application Details to Predict Approval"
)
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77fd74665f3462894e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
